In [ ]:
# imports
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
def load_data():
    """Load the generated data"""
    uploaded_files = files.upload()
    orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
    payments = pd.read_csv('payments.csv', parse_dates=['payment_date'])
    return orders, payments

In [ ]:
def basic_reconciliation(orders, payments):
    """Perform basic order-payment reconciliation"""

    # Clean payments: remove failed transactions and handle nulls
    payments_clean = payments[
        (payments['transaction_status'] == 'success') &
        (payments['amount'] > 0) &
        (payments['amount'].notna())
    ].copy()

    # Aggregate payments by order
    payment_summary = payments_clean.groupby('order_id').agg({
        'amount': 'sum',
        'payment_id': 'count',
        'payment_method': lambda x: ', '.join(x.unique())
    }).rename(columns={'amount': 'total_paid', 'payment_id': 'payment_count'})

    # Merge with orders
    reconciliation = orders.merge(payment_summary, on='order_id', how='left')

    # Fill NaN values for orders with no payments
    reconciliation['total_paid'] = reconciliation['total_paid'].fillna(0)
    reconciliation['payment_count'] = reconciliation['payment_count'].fillna(0)

    # Calculate differences
    reconciliation['difference'] = reconciliation['total_amount'] - reconciliation['total_paid']
    reconciliation['abs_difference'] = abs(reconciliation['difference'])

    # Determine status
    def get_status(row):
        if row['total_paid'] == 0:
            return 'No Payment'
        elif abs(row['difference']) < 0.01:         # Account for floating point
            return 'Fully Paid'
        elif row['difference'] > 0:
            return 'Underpaid'
        else:
            return 'Overpaid'

    reconciliation['status'] = reconciliation.apply(get_status, axis=1)

    return reconciliation

In [ ]:
def generate_report(reconciliation):
    """Create summary report"""

    report = {
        'total_orders': len(reconciliation),
        'total_order_value': reconciliation['total_amount'].sum(),
        'total_paid': reconciliation['total_paid'].sum(),
        'total_difference': reconciliation['difference'].sum(),
        'orders_fully_paid': (reconciliation['status'] == 'Fully Paid').sum(),
        'orders_underpaid': (reconciliation['status'] == 'Underpaid').sum(),
        'orders_overpaid': (reconciliation['status'] == 'Overpaid').sum(),
        'orders_no_payment': (reconciliation['status'] == 'No Payment').sum(),
        'total_underpayment_amount': reconciliation[reconciliation['difference'] > 0]['difference'].sum(),
        'total_overpayment_amount': abs(reconciliation[reconciliation['difference'] < 0]['difference'].sum())
    }

    return pd.DataFrame([report])

In [ ]:
orders, payments = load_data()

Saving orders.csv to orders.csv
Saving payments.csv to payments.csv


In [ ]:
reconciliation = basic_reconciliation(orders, payments)

In [ ]:
reconciliation.head()

,order_id,customer_id,order_date,num_items,subtotal,tax,shipping,total_amount,currency,status,total_paid,payment_count,payment_method,difference,abs_difference
0,ORD-00001,bf3c8405-e80d-4159-a07c-0eedf0a0f2fa,2026-01-23 22:35:32,1,34.76,5.21,6.88,46.85,USD,Fully Paid,46.85,1.0,paypal,0.00,0.00
1,ORD-00002,286281f4-9230-408b-b5b5-7ea752ad6bc8,2026-03-14 16:24:51,1,594.59,89.19,0.79,684.57,USD,Fully Paid,684.57,1.0,bank_transfer,0.00,0.00
2,ORD-00003,da48e822-e195-406f-a645-ccc4f03e19ba,2026-03-28 09:15:52,1,565.63,84.84,17.90,668.37,USD,Underpaid,614.90,2.0,"credit_card, paypal",53.47,53.47
3,ORD-00004,3d4f2ac2-19c6-46a0-8d3e-23c0da653a49,2026-03-02 20:48:31,5,2513.69,377.05,6.95,2897.69,USD,Underpaid,2665.87,1.0,credit_card,231.82,231.82
4,ORD-00005,eba11ed0-9a24-4448-a46a-da1c0ff9c71d,2026-02-15 15:32:37,1,386.13,57.92,8.97,453.02,USD,Overpaid,822.28,2.0,"bank_transfer, crypto",-369.26,369.26


In [ ]:
summary = generate_report(reconciliation)

In [ ]:
summary

,total_orders,total_order_value,total_paid,total_difference,orders_fully_paid,orders_underpaid,orders_overpaid,orders_no_payment,total_underpayment_amount,total_overpayment_amount
0,500,830296.07,682280.43,148015.64,296,60,60,84,176317.82,28302.18


In [ ]:
# Save results
try:
    reconciliation.to_csv('../data/processed/reconciliation_basic.csv', index=False)
    summary.to_csv('../outputs/reports/basic_summary.csv', index=False)
except Exception as e:
    print(e)